In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [6]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.97      0.05
LBP_003_PET                                                      0.00 0.97      0.05
LBP_012_CT                                                       0.00 0.96      0.06
LBP_012_PET                                                      0.01 0.94      0.09
LBP_021_CT                                                       0.00 0.98      0.04
LBP_021_PET                                                      0.01 0.94      0.10
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [7]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [8]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.07
LBP_003_PET                                                      0.08 0.77      0.37
LBP_012_CT                                                       0.00 0.95      0.07
LBP_012_PET                                                      0.18 0.67      0.57
LBP_021_CT                                                       0.15 0.69      0.53
LBP_021_PET                                                      0.01 0.94      0.09
LBP_030_CT                                                       0.00 0.99      0.01
LBP_030_PET       

In [9]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


## Test dataset: MAASTRO 

In [10]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [11]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [12]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [13]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [14]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [15]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [16]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [17]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [18]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [19]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [20]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = StandardScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [23]:
# Divide the X_MAASTRO into numerical part and categorical part 
X_MAASTRO_categorical = X_MAASTRO[categorical_columns]
X_MAASTRO_numeric = X_MAASTRO.drop(categorical_columns, axis=1)

# Save the column name and index of the numeric part
X_MAASTRO_numeric_columns = X_MAASTRO_numeric.columns
X_MAASTRO_numeric_index = X_MAASTRO_numeric.index

In [24]:
# Standardize the numeric part 
X_MAASTRO_numeric_std = scaler.transform(X_MAASTRO_numeric)

# Change the standardized part into a dataframe 
X_MAASTRO_numeric_std = pd.DataFrame(X_MAASTRO_numeric_std, columns=X_MAASTRO_numeric_columns, index=X_MAASTRO_numeric_index)

# Concat the standardized part with the categorical part 
X_MAASTRO_std = pd.concat([X_MAASTRO_categorical, X_MAASTRO_numeric_std], axis=1)

# Change the column order of X_MAASTRO_std
X_MAASTRO_std = X_MAASTRO_std[original_X.columns]

In [25]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 03:21:21,188] A new study created in memory with name: no-name-1eafe64e-80fa-49fe-923c-6250eb9ed44b


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-18 03:21:24,913] Trial 0 failed with parameters: {} because of the following error: ValueError('search direction contains NaN or infinite values').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_5821/4028431732.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 454, in fit
    raise ValueError("search direction contains NaN or infinite values")
ValueError: search direction contains NaN or infinite values
[W 2024-04-18 03:21:24,934] Trial 0 failed with value None.


ValueError: search direction contains NaN or infinite values

In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [28]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [29]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [30]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [32]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [33]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [34]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:21:50,599] A new study created in memory with name: no-name-85fb53e0-215a-4ec0-bc95-d4aaeee61c21


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7303921568627451
Fold 4 C-index: 0.7151898734177216


[I 2024-04-18 03:21:51,717] A new study created in memory with name: no-name-84dd02ae-cf28-4dca-888c-58b25d977e8a


Fold 5 C-index: 0.7488262910798122
[I 2024-04-18 03:21:51,707] Trial 0 finished with value: 0.7118794997698913 and parameters: {}. Best is trial 0 with value: 0.7118794997698913.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7118794997698913], datetime_start=datetime.datetime(2024, 4, 18, 3, 21, 50, 636093), datetime_complete=datetime.datetime(2024, 4, 18, 3, 21, 51, 706705), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7118794997698913


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651743727053
Fold 2 IBS: 0.2215779111122035
Fold 3 IBS: 0.20453593920513286
Fold 4 IBS: 0.22473803623356947
Fold 5 IBS: 0.2181243076466848
[I 2024-04-18 03:21:53,588] Trial 0 finished with value: 0.21659054232697222 and parameters: {}. Best is trial 0 with value: 0.21659054232697222.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054232697222], datetime_start=datetime.datetime(2024, 4, 18, 3, 21, 51, 887444), datetime_complete=datetime.datetime(2024, 4, 18, 3, 21, 53, 587551), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054232697222


In [35]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [36]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.712
train_ibs:  0.217


#### Test

In [37]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [38]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.586


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [39]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:21:54,849] A new study created in memory with name: no-name-347bfd13-7989-4411-a02b-5e003dea5b82


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.4955357142857143
Fold 3 C-index: 0.6911764705882353
Fold 4 C-index: 0.5780590717299579


[I 2024-04-18 03:22:05,383] A new study created in memory with name: no-name-9b4d6b8c-b0a2-4a53-96c3-aecaa1e2f08e


Fold 5 C-index: 0.6197183098591549
[I 2024-04-18 03:22:05,227] Trial 0 finished with value: 0.5920494284441277 and parameters: {}. Best is trial 0 with value: 0.5920494284441277.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5920494284441277], datetime_start=datetime.datetime(2024, 4, 18, 3, 21, 54, 985019), datetime_complete=datetime.datetime(2024, 4, 18, 3, 22, 5, 227131), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5920494284441277


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.3648427575285378
Fold 2 IBS: 0.42988623278732496
Fold 3 IBS: 0.26853768956994967
Fold 4 IBS: 0.4100303033162769
Fold 5 IBS: 0.3361171766744235
[I 2024-04-18 03:22:15,149] Trial 0 finished with value: 0.3618828319753026 and parameters: {}. Best is trial 0 with value: 0.3618828319753026.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.3618828319753026], datetime_start=datetime.datetime(2024, 4, 18, 3, 22, 5, 505386), datetime_complete=datetime.datetime(2024, 4, 18, 3, 22, 15, 148499), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.3618828319753026


In [41]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.592
train_ibs:  0.362


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.541


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.442


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:22:21,293] A new study created in memory with name: no-name-4a814e57-b168-4f5f-9106-23c5b4876d2e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.5223214285714286
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.6033755274261603
Fold 5 C-index: 0.6384976525821596
[I 2024-04-18 03:22:29,251] Trial 0 finished with value: 0.6153421048073681 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6153421048073681.
Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.5580357142857143
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.6371308016877637
Fold 5 C-index: 0.647887323943662
[I 2024-04-18 03:22:34,678] Trial 1 finished with value: 0.6387915675759923 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6387915675759923.
Fold 1 C-index: 0.6406926406926406
Fold 2 C-index: 0.5669642857142857
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.6497890295358649
Fold 5 C-index: 0.6525821596244131
[I 2024-04-18 03:22:40,616] Trial 2 finished with value: 0.644162485858539 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.5580357142857143
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.6371308016877637
Fold 5 C-index: 0.647887323943662
[I 2024-04-18 03:24:30,462] Trial 24 finished with value: 0.6387915675759923 and parameters: {'l1_ratio': 0.28996492542517793}. Best is trial 12 with value: 0.7281116435293545.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.5982142857142857
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6835443037974683
Fold 5 C-index: 0.6572769953051644
[I 2024-04-18 03:24:35,465] Trial 25 finished with value: 0.6653090268182347 and parameters: {'l1_ratio': 0.07330522822697523}. Best is trial 12 with value: 0.7281116435293545.
Fold 1 C-index: 0.6406926406926406
Fold 2 C-index: 0.5669642857142857
Fold 3 C-index: 0.7254901960784313
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.6525821596244131
[I 2024-04-18 03:24:40,629] Trial 26 finished with value: 0.6496353078987475 and parameters: {'l1_ratio': 0.161772298002897

Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.5625
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.6455696202531646
Fold 5 C-index: 0.647887323943662
[I 2024-04-18 03:26:28,801] Trial 48 finished with value: 0.6423525805887923 and parameters: {'l1_ratio': 0.2607230626544604}. Best is trial 12 with value: 0.7281116435293545.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.5357142857142857
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.620253164556962
Fold 5 C-index: 0.6431924882629108
[I 2024-04-18 03:26:35,286] Trial 49 finished with value: 0.6208964134771398 and parameters: {'l1_ratio': 0.4477016529538363}. Best is trial 12 with value: 0.7281116435293545.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.5223214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6160337552742616
Fold 5 C-index: 0.6338028169014085
[I 2024-04-18 03:26:41,341] Trial 50 finished with value: 0.6179151753977009 and parameters: {'l1_ratio': 0.5938078108685914}. Best is tri

...

Fold 1 IBS: 0.3475140168979067
Fold 2 IBS: 0.41225365552385507
Fold 3 IBS: 0.26947453727312354
Fold 4 IBS: 0.38254539171702573
Fold 5 IBS: 0.3114004088282101
[I 2024-04-18 03:30:28,858] Trial 0 finished with value: 0.3446376020480242 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.3446376020480242.
Fold 1 IBS: 0.3154714056953797
Fold 2 IBS: 0.36488183728952245
Fold 3 IBS: 0.2557645179781991
Fold 4 IBS: 0.2914563347249407
Fold 5 IBS: 0.27936066808237664
[I 2024-04-18 03:30:34,437] Trial 1 finished with value: 0.3013869527540837 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.3013869527540837.
Fold 1 IBS: 0.3069941616433161
Fold 2 IBS: 0.3526868276074873
Fold 3 IBS: 0.25055644185398335
Fold 4 IBS: 0.2761780781844828
Fold 5 IBS: 0.2756358845465405
[I 2024-04-18 03:30:40,143] Trial 2 finished with value: 0.292410278767162 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.292410278767162.
Fold 1 

Fold 1 IBS: 0.27670167178087807
Fold 2 IBS: 0.28393290173634916
Fold 3 IBS: 0.2101634344854492
Fold 4 IBS: 0.2375103669674679
Fold 5 IBS: 0.2595429910302284
[I 2024-04-18 03:32:50,349] Trial 25 finished with value: 0.2535702732000745 and parameters: {'l1_ratio': 0.08591305360046901}. Best is trial 21 with value: 0.21603927510328452.
Fold 1 IBS: 0.3087210843663218
Fold 2 IBS: 0.3552070916196057
Fold 3 IBS: 0.2516074332897404
Fold 4 IBS: 0.27912515979376534
Fold 5 IBS: 0.27639597165810637
[I 2024-04-18 03:32:58,260] Trial 26 finished with value: 0.2942113481455079 and parameters: {'l1_ratio': 0.2369889061037231}. Best is trial 21 with value: 0.21603927510328452.
Fold 1 IBS: 0.3562609544733586
Fold 2 IBS: 0.43176165133392436
Fold 3 IBS: 0.26160556353498415
Fold 4 IBS: 0.40132749907226434
Fold 5 IBS: 0.3193968786274274
[I 2024-04-18 03:33:09,290] Trial 27 finished with value: 0.35407050940839174 and parameters: {'l1_ratio': 0.8410447076604634}. Best is trial 21 with value: 0.21603927510328

Fold 1 IBS: 0.33778802019174287
Fold 2 IBS: 0.38741820423743006
Fold 3 IBS: 0.264810289756514
Fold 4 IBS: 0.3331631620960283
Fold 5 IBS: 0.29170859924633696
[I 2024-04-18 03:35:02,186] Trial 50 finished with value: 0.3229776551056104 and parameters: {'l1_ratio': 0.47241766561900594}. Best is trial 21 with value: 0.21603927510328452.
Fold 1 IBS: 0.21371258737198034
Fold 2 IBS: 0.22121706409767253
Fold 3 IBS: 0.20406849034599145
Fold 4 IBS: 0.2244323393541318
Fold 5 IBS: 0.21767845909598596
[I 2024-04-18 03:35:03,871] Trial 51 finished with value: 0.21622178805315243 and parameters: {'l1_ratio': 0.006012844952470507}. Best is trial 21 with value: 0.21603927510328452.
Fold 1 IBS: 0.260653224914918
Fold 2 IBS: 0.24077638305525004
Fold 3 IBS: 0.17468089270319162
Fold 4 IBS: 0.22301401999923057
Fold 5 IBS: 0.24877391215848496
[I 2024-04-18 03:35:09,150] Trial 52 finished with value: 0.22957968656621502 and parameters: {'l1_ratio': 0.03723966440107636}. Best is trial 21 with value: 0.21603927

Fold 5 IBS: 0.2584616505371973
[I 2024-04-18 03:36:42,531] Trial 74 finished with value: 0.2507851430944291 and parameters: {'l1_ratio': 0.07892043023845302}. Best is trial 21 with value: 0.21603927510328452.
Fold 1 IBS: 0.21313182047046095
Fold 2 IBS: 0.22037736475237674
Fold 3 IBS: 0.16010920074470453
Fold 4 IBS: 0.2237320569494287
Fold 5 IBS: 0.23836461838323328
[I 2024-04-18 03:36:45,006] Trial 75 finished with value: 0.21114301226004079 and parameters: {'l1_ratio': 0.02072955188453046}. Best is trial 75 with value: 0.21114301226004079.
Fold 1 IBS: 0.3287530218967289
Fold 2 IBS: 0.37917617700959977
Fold 3 IBS: 0.26158106589941843
Fold 4 IBS: 0.31501595513424246
Fold 5 IBS: 0.2853694495234786
[I 2024-04-18 03:36:50,825] Trial 76 finished with value: 0.31397913389269366 and parameters: {'l1_ratio': 0.3870355975595541}. Best is trial 75 with value: 0.21114301226004079.
Fold 1 IBS: 0.28530342871988335
Fold 2 IBS: 0.3040168330578237
Fold 3 IBS: 0.225551220727241
Fold 4 IBS: 0.2445190322

Fold 1 IBS: 0.25619755604275585
Fold 2 IBS: 0.22004430756398247
Fold 3 IBS: 0.1652534368350093
Fold 4 IBS: 0.223457953868103
Fold 5 IBS: 0.2429068906694106
[I 2024-04-18 03:38:26,163] Trial 99 finished with value: 0.2215720289958522 and parameters: {'l1_ratio': 0.02684879348673712}. Best is trial 88 with value: 0.21108631211765821.


* Best trial for IBS: 
 FrozenTrial(number=88, state=TrialState.COMPLETE, values=[0.21108631211765821], datetime_start=datetime.datetime(2024, 4, 18, 3, 37, 34, 498429), datetime_complete=datetime.datetime(2024, 4, 18, 3, 37, 36, 763722), params={'l1_ratio': 0.020553093448131873}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=88, value=None)


* Best Score for IBS: 
 0.21108631211765821


In [47]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.729
train_ibs:  0.211


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.02638395621436353)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.020553093448131873)

test_ibs:  0.221


In [51]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 03:38:27,556] A new study created in memory with name: no-name-ca1bf25d-a49b-4901-b0b0-eb18e25d48f0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.7383966244725738
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 03:40:01,892] Trial 0 finished with value: 0.7366032203169602 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7366032203169602.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.6854460093896714
[I 2024-04-18 03:40:17,272] Trial 1 finished with value: 0.7181620567966567 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.6038961038961039
Fold 2 C-index: 0.7165178571428571
Fold 3 C-index: 0.7009803921568627
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6032863849765259
[I 2024-04-18 03:47:01,812] Trial 16 finished with value: 0.6633327721070438 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': None, 'min_weight_fraction_leaf': 0.1048543527585735, 'warm_start': False}. Best is trial 12 with value: 0.7438602547617209.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 03:47:21,017] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 417, 'oob_score': True, 'max_samples': 0.1424705672746025, 'max_features': None, 'min_weight_fraction_leaf': 0.20205510510599584, 'warm_start': F

Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.812206572769953
[I 2024-04-18 03:51:23,228] Trial 31 finished with value: 0.7936613975238874 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 215, 'oob_score': True, 'max_samples': 0.7258118850709611, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.020186907613437947, 'warm_start': True}. Best is trial 26 with value: 0.7952997362091728.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.8169014084507042
[I 2024-04-18 03:51:27,117] Trial 32 finished with value: 0.7892741864327053 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 149, 'oob_score': True, 'max_samples': 0.7060378252364139, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0461866753469

...

Fold 1 IBS: 0.20444739845582136
Fold 2 IBS: 0.17446698341767913
Fold 3 IBS: 0.1910994493903993
Fold 4 IBS: 0.1668825886510828
Fold 5 IBS: 0.21103427734544067
[I 2024-04-18 04:17:08,298] Trial 32 finished with value: 0.18958613945208463 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 279, 'oob_score': False, 'max_samples': 0.6325115094238634, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0760629688237641}. Best is trial 21 with value: 0.18758259521522488.
Fold 1 IBS: 0.21394095082970305
Fold 2 IBS: 0.18957239420831615
Fold 3 IBS: 0.18237897227425182
Fold 4 IBS: 0.17846689832832333
Fold 5 IBS: 0.2185435449686763
[I 2024-04-18 04:17:09,958] Trial 33 finished with value: 0.19658055212185413 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 19, 'oob_score': False, 'max_samples': 0.7088178154422675, 'max_features': 'auto', 'min_weight_fraction_leaf':

Fold 1 IBS: 0.21397545241919333
Fold 2 IBS: 0.22060585341756236
Fold 3 IBS: 0.20519466411613427
Fold 4 IBS: 0.2248054198587066
Fold 5 IBS: 0.21758241895615993
[I 2024-04-18 04:21:11,054] Trial 48 finished with value: 0.2164327617535513 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 324, 'oob_score': True, 'max_samples': 0.1840472657347053, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.44962767788541247}. Best is trial 39 with value: 0.18741882102325463.
Fold 1 IBS: 0.20733215663048163
Fold 2 IBS: 0.19840873877308707
Fold 3 IBS: 0.1896647344947966
Fold 4 IBS: 0.2057146078926162
Fold 5 IBS: 0.2152447795126918
[I 2024-04-18 04:21:19,158] Trial 49 finished with value: 0.20327300346073468 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 187, 'oob_score': True, 'max_samples': 0.4611544071310423, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.18697650055709675
Fold 2 IBS: 0.18964751614322145
Fold 3 IBS: 0.17157067951262514
Fold 4 IBS: 0.17048079186700552
Fold 5 IBS: 0.20812563312237994
[I 2024-04-18 04:32:18,923] Trial 64 finished with value: 0.18536022424046578 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 381, 'oob_score': True, 'max_samples': 0.36087148528744895, 'max_features': None, 'min_weight_fraction_leaf': 0.10501492814170305}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.185980269157479
Fold 2 IBS: 0.18929665176523391
Fold 3 IBS: 0.17261558628019577
Fold 4 IBS: 0.17528863593015434
Fold 5 IBS: 0.20987976574068176
[I 2024-04-18 04:33:07,704] Trial 65 finished with value: 0.18661218177474898 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 389, 'oob_score': False, 'max_samples': 0.35503340896452096, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.2139852535910293
Fold 2 IBS: 0.22079135086710738
Fold 3 IBS: 0.2049571432154721
Fold 4 IBS: 0.22449984942587378
Fold 5 IBS: 0.21784420818016068
[I 2024-04-18 04:40:52,063] Trial 80 finished with value: 0.21641556105592863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 401, 'oob_score': False, 'max_samples': 0.23317876663764808, 'max_features': None, 'min_weight_fraction_leaf': 0.13195032778351135}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.18698963922132994
Fold 2 IBS: 0.19117908340974094
Fold 3 IBS: 0.1716425830420473
Fold 4 IBS: 0.1706227695034198
Fold 5 IBS: 0.20890166136319582
[I 2024-04-18 04:41:41,408] Trial 81 finished with value: 0.1858671473079468 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 7, 'min_samples_leaf': 3, 'max_depth': 3, 'n_estimators': 342, 'oob_score': False, 'max_samples': 0.3619448678095568, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.19987889711029083
Fold 2 IBS: 0.18568352389906653
Fold 3 IBS: 0.18380646487398802
Fold 4 IBS: 0.16033813976360883
Fold 5 IBS: 0.20642365546571148
[I 2024-04-18 04:55:41,254] Trial 96 finished with value: 0.18722613622253315 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 363, 'oob_score': False, 'max_samples': 0.4255182801365352, 'max_features': None, 'min_weight_fraction_leaf': 0.08056036420837291}. Best is trial 84 with value: 0.1832464308635407.
Fold 1 IBS: 0.19365837237754394
Fold 2 IBS: 0.183435405665406
Fold 3 IBS: 0.18008910470800357
Fold 4 IBS: 0.17339462295545943
Fold 5 IBS: 0.20781358095494984
[I 2024-04-18 04:56:30,417] Trial 97 finished with value: 0.18767821733227258 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 1, 'n_estimators': 344, 'oob_score': False, 'max_samples': 0.4693979321374093, 'max_features': None, 'min_weight_fraction_leaf': 0

In [53]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.864
train_ibs:  0.183


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [56]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=11, max_features='auto', max_leaf_nodes=15,
                     max_samples=0.9847467855272036, min_samples_leaf=4,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.04749646122417352,
                     n_estimators=230, random_state=123, warm_start=True)

test_cindex:  0.629


RandomSurvivalForest(max_depth=2, max_features=None, max_leaf_nodes=7,
                     max_samples=0.3902292236281065, min_samples_leaf=4,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.0959966989881746,
                     n_estimators=344, random_state=123)

test_ibs:  0.205


In [57]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [59]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 04:57:36,508] A new study created in memory with name: no-name-6ba76334-c459-42d6-99e6-ddaa904995a0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.7605633802816901
[I 2024-04-18 04:57:39,249] Trial 0 finished with value: 0.7784578545189774 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7784578545189774.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:57:46,063] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 04:58:54,024] Trial 16 finished with value: 0.7903520467896392 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 95, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.884183184043622, 'min_weight_fraction_leaf': 0.10581506507448754}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:58:56,391] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 226, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.37217752523657055, 'min_weight_fraction_leaf': 0.1994767874105

Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8860759493670886
Fold 5 C-index: 0.7746478873239436
[I 2024-04-18 04:59:27,290] Trial 31 finished with value: 0.8023701302106281 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 10, 'max_depth': 3, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9175308644810205, 'min_weight_fraction_leaf': 0.1088935662786611}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7652582159624414
[I 2024-04-18 04:59:29,506] Trial 32 finished with value: 0.7950579336258078 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 6, 'min_samples_leaf': 11, 'max_depth': 3, 'n_estimators': 211, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9467713628420997, 'min_weight_fraction_leaf': 0.

...

Fold 1 IBS: 0.18973195855840097
Fold 2 IBS: 0.19901752012782084
Fold 3 IBS: 0.1553163395473765
Fold 4 IBS: 0.13719070606611738
Fold 5 IBS: 0.22779776404493374
[I 2024-04-18 05:17:44,336] Trial 30 finished with value: 0.18181085766892985 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 305, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8220971907615917, 'min_weight_fraction_leaf': 0.076444606879153}. Best is trial 28 with value: 0.18105604370316586.
Fold 1 IBS: 0.2164877455822723
Fold 2 IBS: 0.18827035681808446
Fold 3 IBS: 0.16565039781780372
Fold 4 IBS: 0.1417428766421557
Fold 5 IBS: 0.21575647924950747
[I 2024-04-18 05:18:22,650] Trial 31 finished with value: 0.18558157122196473 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 394, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.91

Fold 1 IBS: 0.21196016007242774
Fold 2 IBS: 0.21737810285668177
Fold 3 IBS: 0.20221519268515586
Fold 4 IBS: 0.21985834792032022
Fold 5 IBS: 0.21707100228614096
[I 2024-04-18 05:21:58,129] Trial 45 finished with value: 0.21369656116414532 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 226, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.5998350649536632, 'min_weight_fraction_leaf': 0.05464331917294846}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.18076053104236317
Fold 2 IBS: 0.2172100382529918
Fold 3 IBS: 0.15472808053026357
Fold 4 IBS: 0.14511587127121353
Fold 5 IBS: 0.2339666086051307
[I 2024-04-18 05:22:08,073] Trial 46 finished with value: 0.18635622594039253 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 185, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.840

Fold 1 IBS: 0.1940353275621495
Fold 2 IBS: 0.19268012360689624
Fold 3 IBS: 0.18110607070067974
Fold 4 IBS: 0.18845548212963192
Fold 5 IBS: 0.2126254989546211
[I 2024-04-18 05:26:38,244] Trial 60 finished with value: 0.1937805005907957 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 12, 'max_depth': 12, 'n_estimators': 305, 'oob_score': True, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.8575527275336559, 'min_weight_fraction_leaf': 0.18982981109623043}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.1881809892682945
Fold 2 IBS: 0.19390719781855054
Fold 3 IBS: 0.1588739037458674
Fold 4 IBS: 0.1367801254919651
Fold 5 IBS: 0.2266847482237021
[I 2024-04-18 05:26:59,622] Trial 61 finished with value: 0.18088539290967592 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 19, 'n_estimators': 362, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7615

Fold 1 IBS: 0.20438643925191893
Fold 2 IBS: 0.1844312624795873
Fold 3 IBS: 0.17002168747641047
Fold 4 IBS: 0.14791525733420058
Fold 5 IBS: 0.21362624602582111
[I 2024-04-18 05:30:41,627] Trial 75 finished with value: 0.18407617851358768 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 296, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6651148142523113, 'min_weight_fraction_leaf': 0.0005666053087619444}. Best is trial 74 with value: 0.18029547957640585.
Fold 1 IBS: 0.19306514020322818
Fold 2 IBS: 0.1842925536567217
Fold 3 IBS: 0.16281493341095712
Fold 4 IBS: 0.15026196851923793
Fold 5 IBS: 0.20907571723485416
[I 2024-04-18 05:30:52,874] Trial 76 finished with value: 0.1799020626049998 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 273, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6121

Fold 1 IBS: 0.19250257566167933
Fold 2 IBS: 0.18261531719008467
Fold 3 IBS: 0.16180974496548703
Fold 4 IBS: 0.14890218044290915
Fold 5 IBS: 0.2130135609453814
[I 2024-04-18 05:33:26,965] Trial 90 finished with value: 0.1797686758411083 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 202, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5935532822743834, 'min_weight_fraction_leaf': 0.007362099342399538}. Best is trial 88 with value: 0.1785581840441121.
Fold 1 IBS: 0.19466401388017032
Fold 2 IBS: 0.18486336673650017
Fold 3 IBS: 0.16588446237479346
Fold 4 IBS: 0.1465134541216928
Fold 5 IBS: 0.21287370502296324
[I 2024-04-18 05:33:35,228] Trial 91 finished with value: 0.18095980042722398 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 197, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5826023

In [60]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.886
train_ibs:  0.179


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=9, max_features=None, max_leaf_nodes=14,
                   max_samples=0.965941994806725, min_samples_split=7,
                   min_weight_fraction_leaf=0.017267854864073284,
                   n_estimators=336, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.662


ExtraSurvivalTrees(max_depth=9, max_features=0.1, max_leaf_nodes=20,
                   max_samples=0.5945245081220258, min_samples_leaf=4,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.006841645987133443,
                   n_estimators=279, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.2


In [64]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 05:34:57,954] A new study created in memory with name: no-name-20fd8a33-03ad-4893-b8ff-07651b076ae8


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 05:36:19,958] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 05:36:58,996] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:00:29,139] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7375077219334873.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:03:19,073] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:35:14,126] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.7577518089481858.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5654008438818565
Fold 5 C-index: 0.5821596244131455
[I 2024-04-18 06:38:28,768] Trial 26 finished with value: 0.5295120936590003 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632,

...

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 09:39:12,999] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 09:39:43,404] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 09:46:15,986] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9998769833869746, 'learning_rate': 0.003967598379054899, 'dropout_rate': 0.17654958266634313, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.8981368148483696, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'auto', 'min_impurity_decrease': 7.140027633149786e-05, 'validation_fraction': 0.6362228335648394, 'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 18, 'max_depth': 1}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 09:46:53,008] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8945386029541793, 'learning_rate': 0.01494075298406419, 'dropout_rate': 0.20

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-18 09:54:08,984] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7865219593038091, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.25073094184082545, 'n_estimators': 410, 'criterion': 'squared_error', 'ccp_alpha': 1.2846220433570537, 'min_weight_fraction_leaf': 0.32348748582358233, 'max_features': 1, 'min_impurity_decrease': 2.1196884133822008e-06, 'validation_fraction': 0.854746843935587, 'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 09:54:31,463] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.74450491092411, 'learning_rate': 0.08426314282635561,

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-18 10:02:00,818] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9047974654661481, 'learning_rate': 0.053489441535809173, 'dropout_rate': 0.15111205819257206, 'n_estimators': 351, 'criterion': 'squared_error', 'ccp_alpha': 0.8515398041309932, 'min_weight_fraction_leaf': 0.21790404115359022, 'max_features': 'auto', 'min_impurity_decrease': 2.7145979471070252e-06, 'validation_fraction': 0.8752408074269059, 'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 13, 'max_depth': 1}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 10:02:45,245] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8838515048108662, 'learning_rate': 0.011504505

Fold 3 IBS: 0.20030580413616672
Fold 4 IBS: 0.21827950204883104
Fold 5 IBS: 0.2160767264619057
[I 2024-04-18 10:11:08,159] Trial 99 finished with value: 0.21234381967629634 and parameters: {'subsample': 0.7984456569855936, 'learning_rate': 0.07921297533010697, 'dropout_rate': 0.36361374787631456, 'n_estimators': 498, 'criterion': 'squared_error', 'ccp_alpha': 0.004455045338579978, 'min_weight_fraction_leaf': 0.23447872920034368, 'max_features': 0.1, 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.4378698142708704, 'min_samples_split': 20, 'max_leaf_nodes': 20, 'min_samples_leaf': 15, 'max_depth': 5}. Best is trial 75 with value: 0.20755403023595909.


* Best trial for IBS: 
 FrozenTrial(number=75, state=TrialState.COMPLETE, values=[0.20755403023595909], datetime_start=datetime.datetime(2024, 4, 18, 9, 52, 18, 716336), datetime_complete=datetime.datetime(2024, 4, 18, 9, 52, 59, 590900), params={'subsample': 0.8646743646205938, 'learning_rate': 0.088434092874841

In [66]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.208


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0339977959383996,
                                 criterion='squared_error',
                                 dropout_rate=0.2075412325353082,
                                 learning_rate=0.010706280861824496,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=3.3602815261835675e-07,
                                 min_samples_leaf=13, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4472167339801619,
                                 n_estimators=445, random_state=123,
                                 subsample=0.9030031356045858,
                                 validation_fraction=0.9350158433232643)

C-index score: 0.618


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008810810992982535,
                                 criterion='squared_error',
                                 dropout_rate=0.1693875032679634,
                                 learning_rate=0.08843409287484169,
                                 max_features='auto', max_leaf_nodes=15,
                                 min_impurity_decrease=5.157320437854681e-06,
                                 min_samples_leaf=16, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29044798941528627,
                                 n_estimators=403, random_state=123,
                                 subsample=0.8646743646205938,
                                 validation_fraction=0.7489142536117352)

IBS: 0.213


In [70]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [71]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [72]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 10:11:27,834] A new study created in memory with name: no-name-46918a9a-8ed8-473f-ac8c-cb2f72667a3d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 10:11:32,183] Trial 0 finished with value: 0.6925028294272052 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 10:11:52,562] Trial 1 finished with value: 0.6915224372703425 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.73708920

Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.755868544600939
[I 2024-04-18 10:15:41,605] Trial 19 finished with value: 0.7083923155803433 and parameters: {'subsample': 0.19816662555708264, 'dropout_rate': 0.34721836740782674, 'n_estimators': 400, 'learning_rate': 0.08909215792368923}. Best is trial 11 with value: 0.7307806313500943.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7464788732394366
[I 2024-04-18 10:15:51,644] Trial 20 finished with value: 0.6970241352671638 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.1874011856184571, 'n_estimators': 258, 'learning_rate': 0.06551866052750378}. Best is trial 11 with value: 0.7307806313500943.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index

Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.7464788732394366
[I 2024-04-18 10:19:56,209] Trial 38 finished with value: 0.7013363578547028 and parameters: {'subsample': 0.23339141148458184, 'dropout_rate': 0.3720991365773877, 'n_estimators': 376, 'learning_rate': 0.05887691176315654}. Best is trial 11 with value: 0.7307806313500943.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7370892018779343
[I 2024-04-18 10:20:14,043] Trial 39 finished with value: 0.6970028095761555 and parameters: {'subsample': 0.4274786379279678, 'dropout_rate': 0.7545555141153719, 'n_estimators': 454, 'learning_rate': 0.06892741183938003}. Best is trial 11 with value: 0.7307806313500943.
Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7598039215686274


...

Fold 2 IBS: 0.17924472193393295
Fold 3 IBS: 0.17484421871598207
Fold 4 IBS: 0.19182158774709396
Fold 5 IBS: 0.18517059625171137
[I 2024-04-18 10:36:20,319] Trial 19 finished with value: 0.18664141445599042 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18664141445599042.
Fold 1 IBS: 0.20385002842262298
Fold 2 IBS: 0.17898541076661173
Fold 3 IBS: 0.177518113027334
Fold 4 IBS: 0.1979575811954832
Fold 5 IBS: 0.18798127523636654
[I 2024-04-18 10:36:22,058] Trial 20 finished with value: 0.18925848172968368 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18664141445599042.
Fold 1 IBS: 0.20069110092838113
Fold 2 IBS: 0.1892544406321917
Fold 3 IBS: 0.18218745670213524
Fold 4 IBS: 0.20090706949452952
Fold 5 IBS: 0.19341752476488996
[I 2024-0

Fold 2 IBS: 0.15485293035522857
Fold 3 IBS: 0.17659534672985133
Fold 4 IBS: 0.19361973277327194
Fold 5 IBS: 0.17914968729881123
[I 2024-04-18 10:37:35,129] Trial 38 finished with value: 0.18464925594798232 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 38 with value: 0.18464925594798232.
Fold 1 IBS: 0.23100212050865962
Fold 2 IBS: 0.17227303252924875
Fold 3 IBS: 0.17930071263516983
Fold 4 IBS: 0.2078210507886513
Fold 5 IBS: 0.178426760992233
[I 2024-04-18 10:37:38,249] Trial 39 finished with value: 0.19376473549079248 and parameters: {'subsample': 0.1669627464432204, 'dropout_rate': 0.11471626893495929, 'n_estimators': 82, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.18464925594798232.
Fold 1 IBS: 0.26293391629537805
Fold 2 IBS: 0.26589096377274146
Fold 3 IBS: 0.21048479246800744
Fold 4 IBS: 0.27413065442020285
Fold 5 IBS: 0.2161561283186412
[I 2024-

Fold 2 IBS: 0.18988711900964794
Fold 3 IBS: 0.18367267394166958
Fold 4 IBS: 0.2230944303345007
Fold 5 IBS: 0.17950851621397682
[I 2024-04-18 10:38:52,034] Trial 57 finished with value: 0.20296648941564147 and parameters: {'subsample': 0.2888110376335854, 'dropout_rate': 0.16382075159292628, 'n_estimators': 81, 'learning_rate': 0.07586937026396535}. Best is trial 41 with value: 0.18274238324469935.
Fold 1 IBS: 0.20634292358637418
Fold 2 IBS: 0.17337935278269065
Fold 3 IBS: 0.17241359892116032
Fold 4 IBS: 0.20871385178528976
Fold 5 IBS: 0.18044495600331506
[I 2024-04-18 10:38:53,941] Trial 58 finished with value: 0.188258936615766 and parameters: {'subsample': 0.23747712467318813, 'dropout_rate': 0.12541109823464971, 'n_estimators': 48, 'learning_rate': 0.06314379319286169}. Best is trial 41 with value: 0.18274238324469935.
Fold 1 IBS: 0.20480041983690422
Fold 2 IBS: 0.20390707070173544
Fold 3 IBS: 0.19195122718180715
Fold 4 IBS: 0.21034367609481147
Fold 5 IBS: 0.2052520861846158
[I 2024

Fold 2 IBS: 0.17890097671682756
Fold 3 IBS: 0.19536496805641376
Fold 4 IBS: 0.21719928241327136
Fold 5 IBS: 0.18349965984241834
[I 2024-04-18 10:40:04,627] Trial 76 finished with value: 0.20269580882165514 and parameters: {'subsample': 0.22021500853091197, 'dropout_rate': 0.1008354138446842, 'n_estimators': 117, 'learning_rate': 0.0643383103937687}. Best is trial 41 with value: 0.18274238324469935.
Fold 1 IBS: 0.24744280167222407
Fold 2 IBS: 0.22862663422765966
Fold 3 IBS: 0.18728302309225137
Fold 4 IBS: 0.25910034960984574
Fold 5 IBS: 0.18874406010330383
[I 2024-04-18 10:40:07,434] Trial 77 finished with value: 0.22223937374105693 and parameters: {'subsample': 0.5460655482305514, 'dropout_rate': 0.30077557647360864, 'n_estimators': 74, 'learning_rate': 0.08757704932309289}. Best is trial 41 with value: 0.18274238324469935.
Fold 1 IBS: 0.20798081275670857
Fold 2 IBS: 0.1734898627587147
Fold 3 IBS: 0.1721656544303873
Fold 4 IBS: 0.18867276746479647
Fold 5 IBS: 0.18207785958983425
[I 202

Fold 2 IBS: 0.17190807267658803
Fold 3 IBS: 0.17273953146414783
Fold 4 IBS: 0.18418892924429342
Fold 5 IBS: 0.17339219691878366
[I 2024-04-18 10:41:21,610] Trial 95 finished with value: 0.1826343479971138 and parameters: {'subsample': 0.10055428912475185, 'dropout_rate': 0.255316498372642, 'n_estimators': 78, 'learning_rate': 0.05738254220979494}. Best is trial 93 with value: 0.18255149870746018.
Fold 1 IBS: 0.22219594223226996
Fold 2 IBS: 0.16732409973069712
Fold 3 IBS: 0.1737554070832707
Fold 4 IBS: 0.19390980198895558
Fold 5 IBS: 0.17438128613481677
[I 2024-04-18 10:41:25,443] Trial 96 finished with value: 0.186313307434002 and parameters: {'subsample': 0.1009094963377387, 'dropout_rate': 0.23698793453295677, 'n_estimators': 101, 'learning_rate': 0.054487916689644846}. Best is trial 93 with value: 0.18255149870746018.
Fold 1 IBS: 0.21781474091983286
Fold 2 IBS: 0.17670930555589828
Fold 3 IBS: 0.17432999009881028
Fold 4 IBS: 0.19666863451181552
Fold 5 IBS: 0.17220960871465493
[I 2024

In [73]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [74]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.731
train_ibs:  0.183


#### Test

In [75]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [76]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10580803679784617,
                                              learning_rate=0.08933703080421439,
                                              n_estimators=324,
                                              random_state=123,
                                              subsample=0.11557957726835853)

C-index score: 0.586


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.20721101622638877,
                                              learning_rate=0.06253585066540067,
                                              n_estimators=64, random_state=123,
                                              subsample=0.10192538954917617)

IBS: 0.232


In [77]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [78]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.886,1.0
Randomsurvivalforest,0.864,2.0
GradientBoosting,0.758,3.0
ComponentwiseGradientBoosting,0.731,4.0
CoxElastic,0.729,5.0
CoxRidge,0.712,6.0
CoxLasso,0.592,7.0


In [79]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.179,1.0
Randomsurvivalforest,0.183,2.5
ComponentwiseGradientBoosting,0.183,2.5
GradientBoosting,0.208,4.0
CoxElastic,0.211,5.0
CoxRidge,0.217,6.0
CoxLasso,0.362,7.0


In [80]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.662,1.0
Randomsurvivalforest,0.629,2.0
GradientBoosting,0.618,3.0
CoxElastic,0.590,4.0
CoxRidge,0.586,5.5
ComponentwiseGradientBoosting,0.586,5.5
CoxLasso,0.541,7.0


In [81]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.200,1.0
Randomsurvivalforest,0.205,2.0
GradientBoosting,0.213,3.0
CoxRidge,0.221,4.5
CoxElastic,0.221,4.5
ComponentwiseGradientBoosting,0.232,6.0
CoxLasso,0.442,7.0


In [82]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/standard/no_selection/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_standard_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [83]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-18
